# 浏览器 Python 环境自检

依次运行下面的单元格，检查 NumPy、SciPy、pandas、scikit-learn、Matplotlib 和 ipywidgets。所有计算发生在你的浏览器中。

In [ ]:
import sys, numpy as np, pandas as pd, scipy, sklearn
import matplotlib
from sklearn.linear_model import LinearRegression
X = np.arange(5).reshape(-1, 1)
y = 3 * X.ravel() + 2
reg = LinearRegression().fit(X, y)
assert np.allclose(reg.predict([[6]]), [20])
print("Python", sys.version.split()[0])
print("numpy", np.__version__, "| pandas", pd.__version__)
print("scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)
print("回归测试通过:", reg.coef_, reg.intercept_)

## ipywidgets 和可交互 Matplotlib

首次运行需要安装浏览器内核中的 Python 包（上面的构建依赖已提供 JupyterLab 前端组件）。

**如何验证滑块：**拖动滑块后，下面的“内核收到”数值应同步变化，证明 Python 端收到浏览器的修改。执行下一格绘图代码后，滑块还应实时控制曲线振幅及标题中的 value；再次运行绘图单元格也应使用当前值，而不是固定的 42。再次运行滑块单元格应保留当前值；只有重启 Python 内核后才恢复初始值 42。


In [ ]:
%pip install -q ipywidgets ipympl

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Only create the widget once per Python kernel. Rerunning this cell reuses
# the existing slider and preserves the value chosen in the browser.
if "_ml_browser_slider" not in globals():
    _ml_browser_slider = widgets.IntSlider(
        min=0, max=100, value=42, description="交互测试"
    )
    _ml_browser_slider_readout = widgets.HTML(value="内核收到：42")
    def _ml_browser_slider_changed(change):
        _ml_browser_slider_readout.value = f"内核收到：{change['new']}"
    _ml_browser_slider.observe(_ml_browser_slider_changed, names="value")
display(widgets.VBox([_ml_browser_slider, _ml_browser_slider_readout]))
print("Python 当前值：", _ml_browser_slider.value)


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# This plot is linked to the IntSlider above, not a fixed sine wave.
# Values 0, 42, 100 produce visibly different amplitudes.
if "_ml_browser_slider" not in globals():
    raise RuntimeError("请先执行上面的 IntSlider 单元格")
if "_ml_browser_plot_observer" in globals():
    _ml_browser_slider.unobserve(_ml_browser_plot_observer, names="value")
if "_ml_browser_plot_fig" in globals():
    plt.close(_ml_browser_plot_fig)

_ml_browser_plot_x = np.linspace(0, 2 * np.pi, 200)
_ml_browser_plot_fig, _ml_browser_plot_ax = plt.subplots()
(_ml_browser_plot_line,) = _ml_browser_plot_ax.plot(
    _ml_browser_plot_x, np.zeros_like(_ml_browser_plot_x), color="tab:blue"
)
_ml_browser_plot_ax.set_xlabel("X")
_ml_browser_plot_ax.set_ylabel("Y = (value / 42) · sin(X)")

def _ml_browser_plot_update(change=None):
    value = int(_ml_browser_slider.value)
    amplitude = value / 42.0
    _ml_browser_plot_line.set_ydata(amplitude * np.sin(_ml_browser_plot_x))
    extent = max(0.25, abs(amplitude) * 1.15)
    _ml_browser_plot_ax.set_ylim(-extent, extent)
    _ml_browser_plot_ax.set_title(f"交互式 Matplotlib · value={value}")
    _ml_browser_plot_fig.canvas.draw_idle()

_ml_browser_plot_observer = _ml_browser_plot_update
_ml_browser_slider.observe(_ml_browser_plot_observer, names="value")
_ml_browser_plot_update()
print(f"绘图使用的 value={_ml_browser_slider.value}（拖动滑块会实时更新图像）")
plt.show()


## TensorFlow.js（真正的浏览器端训练）

这部分由 TensorFlow.js 计算，Python 通过 Pyodide 桥接数据。与 Python TensorFlow 的最大区别是训练需要使用 `await model.fit(...)`。

In [ ]:
from browser_ml.tfjs_bridge import load_tfjs, Dense, Sequential, backend
import numpy as np
tfjs = await load_tfjs()
print("TensorFlow.js 版本:", tfjs.version.tfjs)
print("计算后端:", backend())

In [ ]:
rng = np.random.default_rng(7)
X = rng.uniform(-1, 1, size=(160, 2)).astype("float32")
y = (X[:, 0] + X[:, 1] > 0).astype("float32").reshape(-1, 1)
model = Sequential([Dense(8, "relu", input_shape=(2,)), Dense(1, "sigmoid")])
model.compile(loss="binaryCrossentropy", learning_rate=0.03)
history = await model.fit(X, y, epochs=35, batch_size=16)
pred = model.predict(X)
accuracy = np.mean((pred >= 0.5) == y)
print("TF.js 训练准确率:", round(float(accuracy), 3))
print("损失:", history.get("loss", [])[-3:])
assert accuracy > 0.80, "训练准确率偏低，请尝试增加 epochs"
model.dispose()